In [2]:
import tkinter as tk
from tkinter import messagebox, scrolledtext
import csv
from pathlib import Path

# =====================================
#   CONFIG
# =====================================
CSV_FILENAME = "GIG-logistics.csv"   # Make sure this file is in the same folder as your script

# =====================================
#   LOAD DATA
# =====================================
def load_employees():
    employees = []
    file_path = Path(CSV_FILENAME)
    
    if not file_path.exists():
        messagebox.showerror("File Error", f"Cannot find file: {CSV_FILENAME}\nPlease place it in the same folder.")
        return []
    
    try:
        with open(file_path, encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                employees.append({
                    "name": row["Name"].strip(),
                    "department": row["Department"].strip(),
                    "role": row["Role"].strip(),
                    "employee_id": row["EmployeeID"].strip()
                })
        return employees
    except Exception as e:
        messagebox.showerror("CSV Error", f"Error reading CSV file:\n{str(e)}")
        return []


# =====================================
#   GUI & LOGIC
# =====================================
class EmployeeVerifierApp:
    def __init__(self, root):
        self.root = root
        self.root.title("GIG Logistics Employee Verification")
        self.root.geometry("620x580")
        self.root.resizable(False, False)

        self.employees = load_employees()
        if not self.employees:
            self.root.destroy()
            return

        # ── Header ───────────────────────────────
        tk.Label(root, text="GIG Logistics", font=("Helvetica", 18, "bold"), fg="#1a3c6d").pack(pady=(20, 5))
        tk.Label(root, text="Employee Verification System", font=("Helvetica", 12)).pack()

        # ── Input frame ──────────────────────────
        frame = tk.Frame(root)
        frame.pack(pady=25)

        tk.Label(frame, text="Full Name:", font=("Arial", 11)).grid(row=0, column=0, padx=10, pady=8, sticky="e")
        self.name_entry = tk.Entry(frame, width=35, font=("Arial", 11))
        self.name_entry.grid(row=0, column=1, padx=10, pady=8)

        tk.Label(frame, text="Department:", font=("Arial", 11)).grid(row=1, column=0, padx=10, pady=8, sticky="e")
        self.dept_entry = tk.Entry(frame, width=35, font=("Arial", 11))
        self.dept_entry.grid(row=1, column=1, padx=10, pady=8)

        # ── Buttons ──────────────────────────────
        btn_frame = tk.Frame(root)
        btn_frame.pack(pady=15)

        tk.Button(btn_frame, text="Verify", font=("Arial", 11, "bold"),
                  bg="blue", fg="white", width=12,
                  command=self.verify_employee).pack(side="left", padx=10)

        tk.Button(btn_frame, text="Clear", font=("Arial", 11),
                  bg="#6c757d", fg="white", width=12,
                  command=self.clear_fields).pack(side="left", padx=10)

        # ── Result area ──────────────────────────
        self.result_text = scrolledtext.ScrolledText(root, width=70, height=18,
                                                     font=("Consolas", 10), wrap=tk.WORD)
        self.result_text.pack(padx=20, pady=10, fill="both", expand=False)

        self.result_text.config(state="disabled")

    def clear_fields(self):
        self.name_entry.delete(0, tk.END)
        self.dept_entry.delete(0, tk.END)
        self.result_text.config(state="normal")
        self.result_text.delete("1.0", tk.END)
        self.result_text.config(state="disabled")

    def verify_employee(self):
        name_input = self.name_entry.get().strip()
        dept_input = self.dept_entry.get().strip()

        if not name_input or not dept_input:
            messagebox.showwarning("Input Required", "Please enter both name and department.")
            return

        # Normalize for matching (case insensitive)
        name_lower = name_input.lower()
        dept_lower = dept_input.lower()

        found = None
        colleagues = []

        for emp in self.employees:
            if (emp["name"].lower() == name_lower and
                emp["department"].lower() == dept_lower):
                found = emp
            elif emp["department"].lower() == dept_lower:
                colleagues.append(emp)

        self.result_text.config(state="normal")
        self.result_text.delete("1.0", tk.END)

        if found:
            # Welcome message
            welcome = f"Welcome, {found['name']}!\n"
            welcome += f"Role: {found['role']}\n"
            welcome += f"Employee ID: {found['employee_id']}\n"
            welcome += "─" * 50 + "\n\n"

            self.result_text.insert(tk.END, welcome)

            # Colleagues
            if colleagues:
                self.result_text.insert(tk.END, f"Other members of {found['department']} department:\n\n")
                for col in colleagues:
                    line = f"• {col['name']}  ({col['role']})\n"
                    self.result_text.insert(tk.END, line)
            else:
                self.result_text.insert(tk.END, "You are currently the only member in this department.\n")
        else:
            msg = f"Sorry, no employee named\n'{name_input}'\nwas found in the {dept_input} department.\n\n"
            msg += "Please check your spelling or contact HR."
            self.result_text.insert(tk.END, msg)

        self.result_text.config(state="disabled")


# =====================================
#   MAIN
# =====================================
if __name__ == "__main__":
    root = tk.Tk()
    app = EmployeeVerifierApp(root)
    root.mainloop()